In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   VideoGuard — Audio Moderation Service (Colab + ngrok)                ║
# ║   Faster-Whisper (ASR) + PhoBERT (toxic classification)               ║
# ║   Runtime: GPU T4 recommended  |  Port: 8001                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  CẤU HÌNH — CHỈ SỬA PHẦN NÀY
# ══════════════════════════════════════════════════════════════════════════════
WHISPER_MODEL_SIZE = "medium"          # tiny | base | small | medium | large-v3
PHOBERT_MODEL_PATH = "/content/drive/MyDrive/phobert_model"  # <-- thư mục model PhoBERT trên Drive
NGROK_AUTHTOKEN    = ""               # <-- token tại https://dashboard.ngrok.com
PORT               = 8001
PHOBERT_MAX_LENGTH = 128
PHOBERT_BATCH_SIZE = 32
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

# ── Bước 1: Kiểm tra và fix numpy trước tiên ─────────────────────────────────
# pyvi==0.1.1 dùng pickle với numpy cũ → không tương thích numpy>=2.0
# Phải downgrade numpy==1.26.4 và restart kernel
import numpy as _np_check
_np_major = int(_np_check.__version__.split(".")[0])
if _np_major >= 2:
    print(f"⚠️  Phát hiện numpy {_np_check.__version__} (không tương thích với pyvi).")
    print("🔧 Đang downgrade numpy==1.26.4 và restart kernel...")
    _pip("--force-reinstall", "numpy==1.26.4")
    print("✅ numpy đã được cài lại. Kernel sẽ tự restart sau 2 giây...")
    import time; time.sleep(2)
    os.kill(os.getpid(), 9)   # force-restart kernel — cell sẽ tự chạy lại

print(f"✅ numpy {_np_check.__version__} — OK, tiếp tục cài đặt...")

# ── Bước 2: Cài các thư viện còn lại ─────────────────────────────────────────
print("📦 Đang cài đặt thư viện...")
_pip(
    "fastapi==0.115.5",
    "uvicorn[standard]==0.32.1",
    "python-multipart==0.0.18",
    "faster-whisper==1.1.1",
    "transformers==4.46.3",
    "pyvi==0.1.1",
    "pyngrok",
    "nest_asyncio",
)
print("✅ Cài đặt xong!")

# ── Bước 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Bước 4: Imports ───────────────────────────────────────────────────────────
import logging
import tempfile
import threading

import numpy as np
import torch
import nest_asyncio
nest_asyncio.apply()

from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pyvi import ViTokenizer
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
)
logger = logging.getLogger("audio_moderation")

# ── Bước 5: Pydantic schemas ──────────────────────────────────────────────────
class SentencePrediction(BaseModel):
    text:       str
    label:      str
    label_id:   int
    confidence: float
    scores:     dict

class AudioPredictResponse(BaseModel):
    total_sentences: int
    flagged_count:   int
    overall_label:   str
    sentences:       list

class HealthResponse(BaseModel):
    status:         str
    whisper_loaded: bool
    phobert_loaded: bool
    device:         str

# ── Bước 6: Constants ─────────────────────────────────────────────────────────
LABEL_MAP      = {0: "Clean", 1: "Offensive", 2: "Hate"}
FLAGGED_LABELS = {"Offensive", "Hate"}
LABEL_PRIORITY = {"Hate": 2, "Offensive": 1, "Clean": 0}

# ── Bước 7: Model singletons ──────────────────────────────────────────────────
_whisper   = None
_tokenizer = None
_model     = None
_device    = None

def load_whisper():
    global _whisper
    device       = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    logger.info(f"🎙️  Loading Faster-Whisper '{WHISPER_MODEL_SIZE}' on {device} ({compute_type})...")
    _whisper = WhisperModel(WHISPER_MODEL_SIZE, device=device, compute_type=compute_type)
    logger.info("✅ Whisper loaded")

def load_phobert():
    global _tokenizer, _model, _device
    _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"🧠 Loading PhoBERT from: {PHOBERT_MODEL_PATH}  (device={_device})")
    _tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL_PATH)
    _model = AutoModelForSequenceClassification.from_pretrained(PHOBERT_MODEL_PATH).to(_device)
    _model.eval()
    logger.info(f"✅ PhoBERT loaded (device={_device})")

# ── Bước 8: Core logic ───────────────────────────────────────────────────────
def transcribe_audio(audio_path: str):
    segments, info = _whisper.transcribe(audio_path, beam_size=5, language="vi")
    sentences = [seg.text.strip() for seg in segments if seg.text.strip()]
    logger.info(f"📝 language={info.language} (prob={info.language_probability:.2f}), segments={len(sentences)}")
    return sentences

@torch.no_grad()
def phobert_predict(texts):
    segmented = [ViTokenizer.tokenize(str(t).strip()) for t in texts]
    results = []
    for i in range(0, len(segmented), PHOBERT_BATCH_SIZE):
        batch   = segmented[i : i + PHOBERT_BATCH_SIZE]
        encoded = _tokenizer(
            batch, truncation=True, padding=True,
            max_length=PHOBERT_MAX_LENGTH, return_tensors="pt"
        )
        encoded = {k: v.to(_device) for k, v in encoded.items()}
        probs   = torch.softmax(_model(**encoded).logits, dim=-1).cpu().numpy()
        for prob in probs:
            pid = int(np.argmax(prob))
            results.append({
                "label":      LABEL_MAP[pid],
                "label_id":   pid,
                "confidence": float(prob[pid]),
                "scores":     {
                    "Clean":     float(prob[0]),
                    "Offensive": float(prob[1]),
                    "Hate":      float(prob[2]),
                },
            })
    return results

def worst_label(sentences):
    worst = "Clean"
    for s in sentences:
        if LABEL_PRIORITY.get(s.label, 0) > LABEL_PRIORITY.get(worst, 0):
            worst = s.label
    return worst

# ── Bước 9: Load models ───────────────────────────────────────────────────────
print("\n🚀 Đang load models...")
load_whisper()
load_phobert()
print("✅ Tất cả models đã sẵn sàng!\n")

# ── Bước 10: FastAPI app ──────────────────────────────────────────────────────
app = FastAPI(
    title="VideoGuard Audio Moderation",
    description="Faster-Whisper + PhoBERT: Transcribe & classify Vietnamese audio.",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

@app.get("/health", response_model=HealthResponse, tags=["health"])
def health():
    return HealthResponse(
        status="ok",
        whisper_loaded=_whisper is not None,
        phobert_loaded=_model is not None,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

@app.post("/audio/predict", response_model=AudioPredictResponse, tags=["predict"])
async def predict_audio(file: UploadFile = File(..., description="Audio file WAV/MP3")):
    suffix = os.path.splitext(file.filename or "audio.wav")[1] or ".wav"
    try:
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp.write(await file.read())
            tmp_path = tmp.name
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Could not save audio: {exc}")

    try:
        sentences_text = transcribe_audio(tmp_path)
        if not sentences_text:
            return AudioPredictResponse(
                total_sentences=0, flagged_count=0,
                overall_label="Clean", sentences=[]
            )
        preds   = phobert_predict(sentences_text)
        results = [SentencePrediction(text=txt, **pred) for txt, pred in zip(sentences_text, preds)]
        flagged = sum(1 for s in results if s.label in FLAGGED_LABELS)
        verdict = worst_label(results)
        logger.info(f"[predict] sentences={len(results)} flagged={flagged} verdict={verdict}")
        return AudioPredictResponse(
            total_sentences=len(results),
            flagged_count=flagged,
            overall_label=verdict,
            sentences=results,
        )
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass

# ── Bước 11: Khởi động ngrok + uvicorn ───────────────────────────────────────
from pyngrok import ngrok, conf

if not NGROK_AUTHTOKEN:
    raise ValueError("❌ Hãy điền NGROK_AUTHTOKEN ở phần CẤU HÌNH phía trên!")

conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()  # đóng tunnel cũ nếu có

public_url = ngrok.connect(PORT, "http").public_url
print(f"\n{'='*60}")
print(f"🌐 Public API URL : {public_url}")
print(f"📖 Swagger Docs   : {public_url}/docs")
print(f"❤️  Health Check  : {public_url}/health")
print(f"🔊 Predict Audio  : POST {public_url}/audio/predict")
print(f"{'='*60}\n")

# Chạy uvicorn (blocking — cell này chạy liên tục cho đến khi bạn dừng)
uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")
